In [2]:
import sys
sys.path.append(r"C:\Users\Rakesh\Documents\summer-project\src")

from two_level_mc import * # adjust if your notebook is nested differently
from functions_v2 import *

In [4]:
edges, n_vertices, weights = load_graph(r"../../data/raw/power-US-Grid.mtx")
print(f"n_vertices={n_vertices}, edges={len(edges)}")

✓ Loaded: ../../data/raw/power-US-Grid.mtx
  Vertices : 4941
  Edges    : 6594
  Weighted : no

n_vertices=4941, edges=6594


In [7]:
# 3. Phase 1
A, D, L = build_graph_matrices(edges, n_vertices)

# 4. Phase 2 — sparse from the start, given graph size
lambda_min = compute_lambda_min(L, D)
L_sigma = build_shifted_laplacian(L, D, lambda_min)
print("L_sigma type:", type(L_sigma))

✓ Phase 1 complete: A, D, L built as sparse matrices
  Matrix size : 4941 x 4941
  Degree range: [1, 19]
  Non-zeros in L: 18129

✓ lambda_min = 0.000271
  (eigenvalues found: [0.         0.00027102])
✓ Phase 2 complete: L_sigma built (sigma^2 = 0.25)
  L_sigma type: sparse

L_sigma type: <class 'scipy.sparse._csc.csc_matrix'>


In [10]:
G_nx = nx.Graph()
G_nx.add_edges_from(edges)

In [12]:
def build_grouped_aggregation(G_nx, gamma_in, gamma_out, max_size=10):
    """
    Aggregate gamma_in with gamma_in, gamma_out with gamma_out, and
    interior with interior -- no cross-group merging at all.

    Within each group, independently:
    1. Sort nodes by increasing degree.
    2. For each node, in order:
       a. If it has an unassigned neighbor WITHIN THE SAME GROUP,
          pair them together as a new aggregate.
       b. Otherwise, if it has a neighbor (within the same group)
          already in an aggregate that hasn't hit max_size, join
          that aggregate.
       c. Otherwise, it becomes its own singleton aggregate.

    Parameters
    ----------
    G_nx      : networkx.Graph
    gamma_in  : list of int
    gamma_out : list of int
    max_size  : int -- maximum fine vertices per aggregate

    Returns
    -------
    aggregate_of : dict {fine_vertex: coarse_vertex_id}
    n_coarse     : int -- number of coarse vertices (aggregates)
    """
    gamma_in_set = set(gamma_in)
    gamma_out_set = set(gamma_out)
    interior_set = set(G_nx.nodes()) - gamma_in_set - gamma_out_set

    aggregate_of = {}
    agg_sizes = {}
    next_id = 0

    def process_group(group):
        nonlocal next_id

        for v in sorted(group, key=lambda v: G_nx.degree(v)):
            if v in aggregate_of:
                continue

            # Step (a): look for an unassigned neighbor within the same group
            partner = next(
                (n for n in G_nx.neighbors(v)
                 if n in group and n not in aggregate_of),
                None,
            )
            if partner is not None:
                aggregate_of[v] = next_id
                aggregate_of[partner] = next_id
                agg_sizes[next_id] = 2
                next_id += 1
                continue

            # Step (b): look for a neighbor (same group) already in an
            # aggregate that hasn't hit max_size
            joined = False
            for n in G_nx.neighbors(v):
                if n in group and n in aggregate_of:
                    aid = aggregate_of[n]
                    if agg_sizes[aid] < max_size:
                        aggregate_of[v] = aid
                        agg_sizes[aid] += 1
                        joined = True
                        break
            if joined:
                continue

            # Step (c): singleton
            aggregate_of[v] = next_id
            agg_sizes[next_id] = 1
            next_id += 1

    # Process each group independently
    process_group(gamma_in_set)
    process_group(gamma_out_set)
    process_group(interior_set)

    # Relabel to contiguous 0-indexed coarse vertex IDs
    unique_ids = sorted(set(aggregate_of.values()))
    relabel = {old: new for new, old in enumerate(unique_ids)}
    aggregate_of = {v: relabel[a] for v, a in aggregate_of.items()}
    n_coarse = len(unique_ids)

    return aggregate_of, n_coarse

In [15]:
gamma_in, gamma_out, diameter = find_diameter_endpoints(G_nx, n_sample=5, k_hop=4)

In [18]:
check_boundary_fraction(gamma_in, gamma_out, n_vertices)

gamma_in: 23, gamma_out: 14, n_vertices: 4941
Boundary fraction: 0.7488%
  -> Likely safe for aggregation (comparable to validated successes).


0.00748836267961951

In [23]:
aggregate_of, n_coarse = build_grouped_aggregation(G_nx, gamma_in, gamma_out, max_size=10)

In [25]:
# You've already run:
# aggregate_of, n_coarse = build_grouped_aggregation(G_nx, gamma_in, gamma_out, max_size=10)

# ---- Step 1: check the aggregation is actually usable ----
summary = summarize_aggregation(aggregate_of, n_coarse, gamma_in, gamma_out)

n_coarse: 2056 (2056 aggregates)
Size distribution -- min: 1, max: 10, mean: 2.40
Singletons: 7 (0.3%)
gamma_in_coarse: 11, gamma_out_coarse: 4
Overlap (must be empty): set()
Interior coarse vertices: 2041 (99.3%)


In [27]:
# ---- Step 2: build the coarse graph edges ----
coarse_edges, coarse_contribs = build_coarse_graph_edges(edges, aggregate_of)
print(f"coarse edges: {len(coarse_edges)} (from {len(edges)} fine edges)")

# ---- Step 3: build the fine-to-coarse boundary mapping ----
gamma_in_coarse = sorted(set(aggregate_of[v] for v in gamma_in))
gamma_out_coarse = sorted(set(aggregate_of[v] for v in gamma_out))
overlap = set(gamma_in_coarse) & set(gamma_out_coarse)
print(f"gamma_in_coarse: {len(gamma_in_coarse)}, gamma_out_coarse: {len(gamma_out_coarse)}")
print(f"Overlap (must be empty): {overlap}")

coarse edges: 3260 (from 6594 fine edges)
gamma_in_coarse: 11, gamma_out_coarse: 4
Overlap (must be empty): set()


In [29]:
B = build_incidence_matrix(edges, n_vertices)
factor = sparse_cholesky(L_sigma)

setup = TwoLevelSetup(
    edges, n_vertices, gamma_in, gamma_out,
    coarse_edges, coarse_contribs, n_coarse,
    gamma_in_coarse, gamma_out_coarse,
    B, factor, lambda_min
)

In [42]:
result = run_paired_validation(setup, N=300)

Paired samples: 100%|██████████| 300/300 [00:09<00:00, 30.28sample/s, Q_fine=7.2139, Q_coarse=10.9644] 


N = 300 paired samples
Q_fine   : mean=7.213877  var=2385.803171
Q_coarse : mean=10.964447  var=5566.866596
Q_fine - Q_coarse : mean=-3.750570  var=664.056741
Correlation(Q_fine, Q_coarse): 1.0000
Variance reduction: 3.59x


In [46]:
estimate_direct = two_level_estimate(setup, result, N_coarse_only=2000)

Coarse-only samples: 100%|██████████| 2000/2000 [01:01<00:00, 32.51sample/s, Q_coarse=11.5382]


Coarse-only base estimate (N=2000): 11.538190
Correction term mean (paired samples): -3.750570
Two-level estimate of E[Q_fine]: 7.787620
Direct fine-only mean (for comparison): 7.213877


In [37]:
print(f"Correlation: {result['correlation']:.4f}")
print(f"Variance reduction: {result['variance_reduction']:.2f}x")

Correlation: 1.0000
Variance reduction: 3.82x
